In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from __future__ import annotations

In [5]:
import anndata as ad
from adjustText import adjust_text

from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import cellrank as cr
from cellrank.estimators import GPCCA

import doubletdetection

from fa2 import ForceAtlas2

import gc

import harmonypy as hm

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

import networkx as nx

import numpy as np

import palantir

import pandas as pd

import phate

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk
import os, subprocess

In [6]:
import sys

sys.path.append('..')

from pyfuncs.io import load_full_adata, add_ensembl_ids
from pyfuncs.dropletQC import classify_empty_and_damaged
from pyfuncs.general import preprocessing_adata_sub
from pyfuncs.plot_functions import magma, set_plotting_style, plot_volcano, plot_cell_stats, plot_gene_stats
from pyfuncs.common_vars import BASE_DIR, SEED, CELLBENDER_FIXED_ARGS
set_plotting_style()

In [7]:
from pyfuncs.qc import  MT_CONTIG_MOUSE_REFSEQ, compute_qc_metrics, add_droplet_qc, flag_doublets, qc_embedding, plot_qc_overview, nf_band_report, ambient_top_genes, apply_qc_flags, qc_summary
from pyfuncs.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report
from pyfuncs.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report
from pyfuncs.characterization import subset_and_reprocess, check_marker_dict, population_composition
from pyfuncs.cell_types import DICT_MARKERS_MAJOR_POPULATIONS, DICT_MARKERS_FAP, DICT_MARKERS_KRANOCYTE, DICT_MARKERS_SATELLITE, DICT_MARKERS_TENO, DICT_RENAMING, PALETTE_CELL_TYPE
from pyfuncs.plot_functions import savefig

In [8]:
from datetime import date
TODAY = str(date.today())

DATA_DIR = f"{BASE_DIR}/data/ARAUZO_03/"
FIG_DIR = f"{BASE_DIR}/figures/{TODAY}/"
REFERENCE_DIR = f"{BASE_DIR}/data/public_scRNAseq/common/reference"
GTF = f"{REFERENCE_DIR}/GRCm39_NCBI_GCF_000001635.27/GCF_000001635.27_GRCm39_genomic.gtf"

LEIDEN = dict(flavor="igraph", n_iterations=2, directed=False,
              random_state=SEED, neighbors_key="neighbors_harmony")

SEED = 10

# Data loading and QC

In [ ]:
sample_df = pd.read_csv(f"{DATA_DIR}/samples.tsv", sep="\t")
sample_df

In [ ]:
adatas = {}

for idx, row in sample_df.iterrows():
    gsm, condition = row["gsm"], row["condition"]
    print(f"Running preprocessing on {gsm} - {condition}")

    solo   = f"{DATA_DIR}/results_STAR/{gsm}/Solo.out"
    cb_dir = f"{solo}/CellBender"
    cb_in  = f"{solo}/adata_raw.h5ad"            # .h5ad, no .h5
    cb_out = f"{cb_dir}/adata_raw_cellbender.h5"
    os.makedirs(cb_dir, exist_ok=True)

    # ---- Phase 1: Load raw count matrix into CellBender ----
    if not os.path.exists(cb_in):
        a = load_full_adata(solo, include_cellbender=False,
                            load_velocyto=False, keep_raw_layer=False)
        a.layers.clear()                          # CellBender solo mira X
        a.write_h5ad(cb_in)
        del a

    # ---- CellBender ----
    if not os.path.exists(cb_out):
        print(f"[{gsm}] CellBender...")
        subprocess.run([
            "docker", "run", "--gpus", "all", "--rm",
            "-v", f"{DATA_DIR}:{DATA_DIR}",
            "-w", cb_dir,                         
            "us.gcr.io/broad-dsde-methods/cellbender:0.3.2",
            "cellbender", "remove-background",
            "--input", cb_in,
            "--output", cb_out,
            *CELLBENDER_FIXED_ARGS,
        ], check=True)

    # ---- Phase 2: CellBender + Velocyto ----
    adata_idx = load_full_adata(
        solo,
        include_cellbender=True,
        cellbender_h5=cb_out,
        load_velocyto=True,
        keep_raw_layer=True,
        min_cell_probability=0.5,
        gtf_path=GTF,
        cellbender_meta = {"version": "0.3.2", "args": list(CELLBENDER_FIXED_ARGS)},
    )
    
    
    adata_idx.obs["gsm"] = gsm
    adata_idx.obs["condition"] = condition
    adata_idx.write_h5ad(f"{solo}/adata_cellbender_velocyto.h5ad")
    adatas[gsm] = adata_idx

In [ ]:
adatas = {}
for idx, row in sample_df.iterrows():
    gsm = row["gsm"]
    adata = ad.read_h5ad(f"{DATA_DIR}/results_STAR/{gsm}/Solo.out/adata_cellbender_velocyto.h5ad")

    compute_qc_metrics(adata, mt_contig=MT_CONTIG_MOUSE_REFSEQ)
    add_droplet_qc(adata, nf_rescue=0.02, umi_rescue=3000,
                          nf_damaged_threshold=0.5, prep="cell")   
    flag_doublets(adata)                                         
    qc_embedding(adata) 

    plot_cell_stats(adata)
    plot_gene_stats(adata)
    plot_qc_overview(adata)   
    plt.show()
                                    
    adatas[gsm] = adata

In [ ]:
THRESHOLDS = {
    "AZ7845": dict(
        max_pct_mt=15,         
        min_log1p_genes=6.5,
        droplet_exclude=('empty_droplet', 'damaged_cell',),         
        doublet_cols=("scrublet_doublet", "dd_doublet",),
    ),
    "AZ7846": dict(
        max_pct_mt=15,         
        min_log1p_genes=6.5,
        droplet_exclude=('empty_droplet', 'damaged_cell',),         
        doublet_cols=("scrublet_doublet", "dd_doublet",),
    )
}
for gsm, t in THRESHOLDS.items():
    apply_qc_flags(adatas[gsm], **t)

qc_summary(adatas)

In [ ]:
from pyfuncs.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report

In [ ]:
adata = concat_samples(adatas, sample_key="gsm")
adata = adata[adata.obs["qc_pass"]].copy()      # filtrado definitivo

preliminary_clusters(adata, seed=SEED)                      # para el pooling
scran_size_factors(adata, cluster_key="scran_clusters", do_plot=True)
size_factor_report(adata)
apply_size_factors(adata)                        # deja tres layers, no toca X
compare_normalizations(adata, group_key="gsm")

In [ ]:
adata.X = adata.layers["norm_scran_log1p"].copy()
adata.write_h5ad(f"{DATA_DIR}/adata_normalized.h5ad")

In [ ]:
adata = sc.read(f"{DATA_DIR}/adata_normalized.h5ad")

In [ ]:
from pyfuncs.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report

In [ ]:
from pyfuncs.characterization import subset_and_reprocess, check_marker_dict, population_composition

In [ ]:
sc.pp.filter_genes(adata, min_cells=5)
select_hvgs_preliminary(adata, n_top_genes=3000)                  # andamio
build_embeddings(adata, "hvg_prelim", batch_key="gsm", seed=SEED)            # PCA→harmony→neighbors
select_hvgs_triku(adata, n_features=2000)                         # HVGs definitivos
build_embeddings(adata, "hvg_triku", batch_key="gsm", n_neighbors=40, seed=SEED)             # las dos vías finales

soup_vs_hvg_report(adata)
rep, mat = harmony_merge_report(adata, condition_key="gsm")

sns.heatmap(mat/mat.sum(0))

In [ ]:
sc.tl.leiden(adata, resolution=0.1, key_added='leiden', neighbors_key = "neighbors_harmony")
sc.tl.leiden(adata, resolution=3,  key_added='leiden_sub', neighbors_key = "neighbors_harmony")

sc.tl.umap(adata, neighbors_key = "neighbors_harmony", random_state=SEED)


In [ ]:
sc.pl.umap(adata, color=['leiden', 'leiden_sub'])

In [ ]:
# The FACs was done as PDPN(+) CD31/Pecam1(-)

sc.pl.umap(adata, color=['Pdpn', 'Pecam1', 'Pdgfra', 'Tnmd', 'Lum', 'Prg4'], cmap=magma, use_raw=False)

## Analysis of major populations

In [ ]:
assign_cats(
    adata,
    DICT_MARKERS_MAJOR_POPULATIONS,
    column_groupby='leiden_sub', 
    key_added='major_population',
    quantile_gene_sel=0.9,
    min_score=0.3, 
)

In [ ]:
sc.pl.umap(adata, color=['major_population'], cmap=magma, use_raw=False)

In [ ]:
sc.pl.umap(adata, color=['Pdpn', 'Pax7'], cmap=magma, use_raw=False)

## Analysing Tenocyte populations

In [ ]:
adata_teno = subset_and_reprocess(adata, "major_population", "Tenocyte")
_ = check_marker_dict(adata_teno, DICT_MARKERS_TENO)

sc.tl.leiden(adata_teno, resolution=0.1, key_added="leiden",     **LEIDEN)
sc.tl.leiden(adata_teno, resolution=3.0, key_added="leiden_sub", **LEIDEN)
sc.tl.umap(adata_teno, neighbors_key="neighbors_harmony", min_dist=0.3, random_state=SEED)

In [ ]:
assign_cats(
    adata_teno,
    DICT_MARKERS_TENO,
    column_groupby='leiden_sub', 
    key_added='minor_population',
    quantile_gene_sel=0.99,
    min_score=0.3, 
)

sc.pl.umap(adata_teno, color=['minor_population'], cmap=magma, use_raw=False, s=10)

In [ ]:
for pop, genes in DICT_MARKERS_TENO.items():
    sc.pl.umap(adata_teno, color=genes, cmap=magma)

## Analysing Satellite populations

In [ ]:
adata_satellite = subset_and_reprocess(adata, "major_population", "Satellite")
_ = check_marker_dict(adata_satellite, DICT_MARKERS_SATELLITE)

sc.tl.leiden(adata_satellite, resolution=0.1, key_added="leiden",     **LEIDEN)
sc.tl.leiden(adata_satellite, resolution=3.0, key_added="leiden_sub", **LEIDEN)
sc.tl.umap(adata_satellite, neighbors_key="neighbors_harmony", min_dist=0.3, random_state=SEED)

In [ ]:
assign_cats(
    adata_satellite,
    DICT_MARKERS_SATELLITE,
    column_groupby='leiden_sub', 
    key_added='minor_population',
    quantile_gene_sel=0.8,
    min_score=0.1, 
)

sc.pl.umap(adata_satellite, color=['minor_population'], cmap=magma, use_raw=False)

In [ ]:
for pop, genes in DICT_MARKERS_SATELLITE.items():
    sc.pl.umap(adata_satellite, color=[i for i in genes if i in adata_satellite.var_names], cmap=magma)

## Analysing Kranocyte populations

In [ ]:
adata_kranocyte = subset_and_reprocess(adata, "major_population", "Kranocyte")
_ = check_marker_dict(adata_kranocyte, DICT_MARKERS_KRANOCYTE)

sc.tl.leiden(adata_kranocyte, resolution=0.1, key_added="leiden",     **LEIDEN)
sc.tl.leiden(adata_kranocyte, resolution=3.0, key_added="leiden_sub", **LEIDEN)
sc.tl.umap(adata_kranocyte, neighbors_key="neighbors_harmony", min_dist=0.3, random_state=SEED)

In [ ]:
assign_cats(
    adata_kranocyte,
    DICT_MARKERS_KRANOCYTE,
    column_groupby='leiden_sub', 
    key_added='minor_population',
    quantile_gene_sel=0.99,
    min_score=0.3, 
)

sc.pl.umap(adata_kranocyte, color=['minor_population'], cmap=magma, use_raw=False, s=10)

In [ ]:
for pop, genes in DICT_MARKERS_KRANOCYTE.items():
    sc.pl.umap(adata_kranocyte, color=genes, cmap=magma)

## Analysing FAP populations

In [ ]:
adata_FAP = subset_and_reprocess(adata, "major_population", "FAP")
_ = check_marker_dict(adata_FAP, DICT_MARKERS_FAP)

sc.tl.leiden(adata_FAP, resolution=0.1, key_added="leiden",     **LEIDEN)
sc.tl.leiden(adata_FAP, resolution=3.0, key_added="leiden_sub", **LEIDEN)
sc.tl.umap(adata_FAP, neighbors_key="neighbors_harmony", min_dist=0.3, random_state=SEED)

In [ ]:
assign_cats(
    adata_FAP,
    DICT_MARKERS_FAP,
    column_groupby='leiden_sub', 
    key_added='minor_population',
    quantile_gene_sel=0.99,
    min_score=0.3, 
)

sc.pl.umap(adata_FAP, color=['minor_population'], cmap=magma, use_raw=False, s=10)

In [ ]:
for pop, genes in DICT_MARKERS_FAP.items():
    sc.pl.umap(adata_FAP, color=[i for i in genes if i in adata_FAP.var_names], cmap=magma)

# Join back minor populations to main adata and relabelling

In [ ]:
adata.obs['minor_population'] = adata.obs['major_population'].astype(str).values

In [ ]:
for adata_sub in adata_FAP, adata_kranocyte, adata_satellite, adata_teno:
    cells = adata_sub.obs_names
    adata.obs.loc[cells, 'minor_population'] = adata_sub.obs.loc[cells, 'minor_population'].astype(str).values

adata.obs["minor_population"] = adata.obs["minor_population"].replace({"Endothelial": "Endothelial*"})

In [ ]:
sc.pl.umap(adata, color="minor_population")

In [ ]:
adata.obs["cell_type"] = adata.obs["minor_population"].astype(str).replace(DICT_RENAMING)
adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")
adata.uns["cell_type_colors"] = [PALETTE_CELL_TYPE[i] for i in adata.obs["cell_type"].cat.categories]


fig, ax = plt.subplots(1, 1)
sc.pl.umap(adata, color="cell_type", ax=ax, frameon=False, title='', show=False)
savefig(fig=fig, filename="0AA_umap_all-cell-types", fig_dir=FIG_DIR)

In [ ]:
adata.write_h5ad(f"{DATA_DIR}/processed_adatas/AA_inhouse_dataset_processed.h5ad")

## Separating FAPs and plotting their distribution

In [ ]:
adata_FAP = subset_and_reprocess(adata, "major_population", ["FAP", "Kranocyte"], seed=SEED)
sc.tl.umap(adata_FAP, neighbors_key="neighbors_harmony", min_dist=0.3, random_state=SEED)

In [ ]:
fig, ax = plt.subplots(1, 1)
sc.pl.umap(adata_FAP, color="cell_type", ax=ax, frameon=False, title='', show=False)
savefig(fig=fig, filename="0AA_umap_FAPs", fig_dir=FIG_DIR)

In [ ]:
adata_FAP.write_h5ad(f"{DATA_DIR}/processed_adatas/AA_inhouse_dataset_processed_FAPs.h5ad")